# Curasao module tests

基于 `src/datasets/SeathruNeRF_dataset/Curasao` 对 `utils/` 和 `modules/` 的核心功能做轻量测试。默认不写仓库文件，PLY 测试只写系统临时目录。`renderer` 是可选长耗时 CUDA 烟测，默认跳过；需要时在运行 notebook 前设置 `RUN_RENDERER_TEST=1`。

In [26]:
from __future__ import annotations

import importlib.util
import os
import sys
import tempfile
import traceback
from pathlib import Path

import numpy as np
import torch

for module_name in list(sys.modules):
    if module_name == "utils" or module_name.startswith("utils.") or module_name == "modules" or module_name.startswith("modules."):
        del sys.modules[module_name]

DATA_ROOT = Path("src/datasets/SeathruNeRF_dataset/Curasao")
SPARSE_ROOT = DATA_ROOT / "sparse" / "0"
IMAGE_PATH = DATA_ROOT / "images_wb" / "MTN_1288.png"

RESULTS = []


class SkipTest(Exception):
    pass


def assert_true(condition, message):
    if not condition:
        raise AssertionError(message)


def assert_close(actual, expected, atol=1e-6, rtol=1e-6, message="values are not close"):
    if not np.allclose(actual, expected, atol=atol, rtol=rtol):
        raise AssertionError(message)


def run_test(name, fn):
    try:
        detail = fn()
    except SkipTest as exc:
        RESULTS.append((name, "SKIP", str(exc)))
        print(f"SKIP {name}: {exc}")
    except Exception:
        detail = traceback.format_exc()
        RESULTS.append((name, "FAIL", detail))
        print(f"FAIL {name}\n{detail}")
    else:
        RESULTS.append((name, "PASS", "" if detail is None else str(detail)))
        print(f"PASS {name}" + (f": {detail}" if detail else ""))


def require_dataset():
    assert_true(DATA_ROOT.exists(), f"missing dataset: {DATA_ROOT}")
    assert_true(SPARSE_ROOT.exists(), f"missing COLMAP sparse model: {SPARSE_ROOT}")
    assert_true(IMAGE_PATH.exists(), f"missing test image: {IMAGE_PATH}")


require_dataset()
print("DATA_ROOT =", DATA_ROOT)
print("torch =", torch.__version__, "cuda =", torch.cuda.is_available())

DATA_ROOT = src/datasets/SeathruNeRF_dataset/Curasao
torch = 2.11.0+cu126 cuda = True


## Shared Curasao fixtures

In [27]:
from utils.colmap_reader import read_colmap_model
from utils.dataset_loaders import load_colmap_dataset, load_llff_dataset

COLMAP_CAMERAS, COLMAP_IMAGES, COLMAP_POINTS = read_colmap_model(str(SPARSE_ROOT))
COLMAP_SCENE = load_colmap_dataset(str(DATA_ROOT), load_images=False)
LLFF_SCENE = load_llff_dataset(str(DATA_ROOT), load_images=False)

POINT_COUNT = min(128, len(COLMAP_SCENE.point_cloud_xyz))
TEST_XYZ = COLMAP_SCENE.point_cloud_xyz[:POINT_COUNT]
TEST_RGB = COLMAP_SCENE.point_cloud_rgb[:POINT_COUNT]

print("COLMAP:", len(COLMAP_CAMERAS), "camera,", len(COLMAP_IMAGES), "images,", len(COLMAP_POINTS), "points")
print("train scene:", len(COLMAP_SCENE.image_paths), COLMAP_SCENE.c2w_matrices.shape, COLMAP_SCENE.point_cloud_xyz.shape)
print("llff scene:", len(LLFF_SCENE.image_paths), LLFF_SCENE.c2w_matrices.shape, LLFF_SCENE.point_cloud_xyz.shape)

COLMAP: 1 camera, 21 images, 25837 points
train scene: 18 (18, 4, 4) (25837, 3)
llff scene: 18 (18, 4, 4) (25837, 3)


## utils/ tests

In [28]:
def test_syntax_utils():
    for path in sorted(Path("utils").glob("*.py")):
        compile(path.read_text(), str(path), "exec")


def test_colmap_reader():
    from utils.colmap_reader import (
        read_cameras_binary,
        read_cameras_text,
        read_images_binary,
        read_images_text,
        read_points3d_binary,
        read_points3d_text,
    )

    assert_true(len(COLMAP_CAMERAS) == 1, "Curasao should contain one COLMAP camera")
    assert_true(len(COLMAP_IMAGES) == 21, "Curasao should contain 21 COLMAP images")
    assert_true(len(COLMAP_POINTS) > 0, "Curasao COLMAP point cloud should be non-empty")
    camera = next(iter(COLMAP_CAMERAS.values()))
    image = next(iter(COLMAP_IMAGES.values()))
    point = next(iter(COLMAP_POINTS.values()))
    assert_true(camera.width > 0 and camera.height > 0, "camera resolution must be positive")
    assert_true(image.qvec.shape == (4,) and image.tvec.shape == (3,), "image pose shape is wrong")
    assert_true(point.xyz.shape == (3,) and point.rgb.shape == (3,), "point shape is wrong")
    image_names = sorted(image.name for image in COLMAP_IMAGES.values())
    expected_names = sorted(path.name for path in (DATA_ROOT / "images_wb").glob("*.png"))
    assert_true(image_names == expected_names, "COLMAP image names must match images_wb files exactly")
    assert_true(camera.model in {"PINHOLE", "SIMPLE_PINHOLE", "SIMPLE_RADIAL", "RADIAL", "OPENCV"}, "unexpected COLMAP camera model")
    assert_true(0 <= int(point.rgb.min()) <= int(point.rgb.max()) <= 255, "point RGB must be uint8 color data")
    assert_true(len(read_cameras_binary(str(SPARSE_ROOT / "cameras.bin"))) == len(COLMAP_CAMERAS), "read_cameras_binary count mismatch")
    assert_true(len(read_images_binary(str(SPARSE_ROOT / "images.bin"))) == len(COLMAP_IMAGES), "read_images_binary count mismatch")
    assert_true(len(read_points3d_binary(str(SPARSE_ROOT / "points3D.bin"))) == len(COLMAP_POINTS), "read_points3d_binary count mismatch")
    with tempfile.TemporaryDirectory(prefix="sjtu_colmap_text_test_") as text_tmp:
        text_dir = Path(text_tmp)
        (text_dir / "cameras.txt").write_text(f"{camera.camera_id} {camera.model} {camera.width} {camera.height} " + " ".join(map(str, camera.params.tolist())) + "\n")
        first_xy = image.xys[:2]
        first_ids = image.point3d_ids[:2]
        points_line = " ".join(f"{xy[0]} {xy[1]} {pid}" for xy, pid in zip(first_xy, first_ids))
        (text_dir / "images.txt").write_text(" ".join(map(str, [image.image_id, *image.qvec.tolist(), *image.tvec.tolist(), image.camera_id, image.name])) + "\n" + points_line + "\n")
        (text_dir / "points3D.txt").write_text(" ".join(map(str, [point.point3d_id, *point.xyz.tolist(), *point.rgb.tolist(), point.error])) + " 1 2\n")
        text_cameras = read_cameras_text(str(text_dir / "cameras.txt"))
        text_images = read_images_text(str(text_dir / "images.txt"))
        text_points = read_points3d_text(str(text_dir / "points3D.txt"))
        model_text_cameras, model_text_images, model_text_points = read_colmap_model(str(text_dir))
    assert_close(text_cameras[camera.camera_id].params, camera.params, message="read_cameras_text params mismatch")
    assert_close(text_images[image.image_id].qvec, image.qvec, message="read_images_text qvec mismatch")
    assert_close(text_images[image.image_id].xys, first_xy, message="read_images_text xys mismatch")
    assert_close(text_points[point.point3d_id].xyz, point.xyz, message="read_points3d_text xyz mismatch")
    assert_true(len(model_text_cameras) == 1 and len(model_text_images) == 1 and len(model_text_points) == 1, "read_colmap_model text fallback failed")
    return f"{len(COLMAP_IMAGES)} images, {len(COLMAP_POINTS)} points"


def test_dataset_loaders():
    from utils.camera_utils import colmap_image_to_c2w

    for name, scene in [("colmap", COLMAP_SCENE), ("llff", LLFF_SCENE)]:
        assert_true(len(scene.image_paths) == 18, f"{name} train split should contain 18 images")
        assert_true(scene.c2w_matrices.shape == (18, 4, 4), f"{name} c2w shape is wrong")
        assert_true(scene.fx.shape == (18,) and scene.fy.shape == (18,), f"{name} focal arrays are wrong")
        assert_true(scene.cx.shape == (18,) and scene.cy.shape == (18,), f"{name} principal point arrays are wrong")
        assert_true(scene.width > 0 and scene.height > 0, f"{name} resolution must be positive")
        assert_true(0 < scene.near < scene.far, f"{name} near/far must be ordered")
        assert_true(scene.scene_extent > 0, f"{name} scene_extent must be positive")
        assert_true(scene.point_cloud_xyz is not None and scene.point_cloud_xyz.shape[1] == 3, f"{name} xyz shape is wrong")
        assert_true(scene.point_cloud_rgb is not None and scene.point_cloud_rgb.shape[1] == 3, f"{name} rgb shape is wrong")
        assert_true(all(Path(path).exists() for path in scene.image_paths), f"{name} image path missing")
    sorted_ids = sorted(COLMAP_IMAGES.keys(), key=lambda item: COLMAP_IMAGES[item].name)
    train_ids = [image_id for i, image_id in enumerate(sorted_ids) if i % 8 != 0]
    expected_paths = [str(DATA_ROOT / "images_wb" / COLMAP_IMAGES[image_id].name) for image_id in train_ids]
    expected_c2w = np.stack([colmap_image_to_c2w(COLMAP_IMAGES[image_id].qvec, COLMAP_IMAGES[image_id].tvec, opengl=True) for image_id in train_ids])
    camera = next(iter(COLMAP_CAMERAS.values()))
    assert_true(COLMAP_SCENE.image_paths == expected_paths, "COLMAP loader train split paths do not match expected holdout rule")
    assert_close(COLMAP_SCENE.c2w_matrices, expected_c2w, atol=1e-8, message="COLMAP loader c2w matrices do not match qvec/tvec conversion")
    assert_close(COLMAP_SCENE.fx, np.full(18, camera.fx), message="COLMAP fx values do not match camera params")
    assert_close(COLMAP_SCENE.fy, np.full(18, camera.fy), message="COLMAP fy values do not match camera params")
    assert_close(COLMAP_SCENE.cx, np.full(18, camera.cx), message="COLMAP cx values do not match camera params")
    assert_close(COLMAP_SCENE.cy, np.full(18, camera.cy), message="COLMAP cy values do not match camera params")
    assert_close(COLMAP_SCENE.point_cloud_xyz[0], next(iter(COLMAP_POINTS.values())).xyz.astype(np.float32), message="COLMAP point xyz payload changed")
    assert_close(COLMAP_SCENE.point_cloud_rgb[0], next(iter(COLMAP_POINTS.values())).rgb, message="COLMAP point rgb payload changed")
    poses_bounds = np.load(DATA_ROOT / "poses_bounds.npy")
    assert_close(LLFF_SCENE.near, float(poses_bounds[:, 15].min() * 0.9), message="LLFF near does not match poses_bounds")
    assert_close(LLFF_SCENE.far, float(poses_bounds[:, 16].max() * 1.1), message="LLFF far does not match poses_bounds")
    colmap_test = load_colmap_dataset(str(DATA_ROOT), split="test", load_images=False)
    llff_val = load_llff_dataset(str(DATA_ROOT), split="val", load_images=False)
    assert_true(len(colmap_test.image_paths) == 3, "COLMAP test split should contain holdout images 0, 8, 16")
    assert_true(len(llff_val.image_paths) == 3, "LLFF val split should contain holdout images 0, 8, 16")
    loaded_scene = load_colmap_dataset(str(DATA_ROOT), split="test", load_images=True)
    assert_true(loaded_scene.images.shape == (3, COLMAP_SCENE.height, COLMAP_SCENE.width, 3), "load_images=True image stack shape is wrong")
    return f"colmap={COLMAP_SCENE.width}x{COLMAP_SCENE.height}, llff={LLFF_SCENE.width}x{LLFF_SCENE.height}"


def test_image_utils():
    from utils.image_utils import compute_psnr, compute_ssim, load_image, save_image
    from PIL import Image

    image = load_image(str(IMAGE_PATH), as_float=True)
    small = load_image(str(IMAGE_PATH), as_float=True, resize=(64, 48))
    raw = load_image(str(IMAGE_PATH), as_float=False, resize=(32, 24))
    assert_true(image.dtype == np.float32, "float image dtype should be float32")
    assert_true(image.ndim == 3 and image.shape[-1] == 3, "image must be HWC RGB")
    assert_true(0.0 <= float(image.min()) <= float(image.max()) <= 1.0, "float image range must be [0, 1]")
    assert_true(small.shape == (48, 64, 3), "resized float image shape is wrong")
    assert_true(raw.dtype == np.uint8 and raw.shape == (24, 32, 3), "uint8 resized image is wrong")
    pil_raw = np.asarray(Image.open(IMAGE_PATH).convert("RGB"))
    assert_close(image, pil_raw.astype(np.float32) / 255.0, atol=0.0, rtol=0.0, message="float image must equal PIL RGB / 255")
    assert_close(raw, np.asarray(Image.open(IMAGE_PATH).convert("RGB").resize((32, 24), Image.Resampling.LANCZOS)), atol=0.0, rtol=0.0, message="resized uint8 image must match PIL LANCZOS output")
    assert_true(compute_psnr(image, image) >= 119.0, "identical image PSNR should be very high")
    changed = image.copy()
    changed[:4, :4] = 0.0
    mask = np.zeros(image.shape[:2], dtype=bool)
    mask[:4, :4] = True
    assert_true(compute_psnr(changed, image, mask=mask) < compute_psnr(changed, image), "masked PSNR should focus on changed pixels")
    save_path = Path(tempfile.gettempdir()) / "sjtu_saved_image_test.png"
    save_image(str(save_path), small)
    saved = load_image(str(save_path), as_float=False)
    assert_close(saved, np.clip(small * 255.0, 0, 255).astype(np.uint8), atol=1, message="save_image PNG payload mismatch")
    try:
        assert_true(compute_ssim(small, small) > 0.999, "identical image SSIM should be near 1")
    except ImportError as exc:
        raise SkipTest(str(exc)) from exc
    return str(image.shape)


def test_camera_utils():
    from utils.camera_utils import (
        average_pose_center,
        c2w_to_viewmat,
        colmap_image_to_c2w,
        estimate_scene_extent,
        intrinsic_matrix,
        focal_to_fov,
        fov_to_focal,
        qvec_to_rotmat,
        rotmat_to_qvec,
    )

    image = next(iter(COLMAP_IMAGES.values()))
    camera = COLMAP_CAMERAS[image.camera_id]
    rot = qvec_to_rotmat(image.qvec)
    qvec2 = rotmat_to_qvec(rot)
    rot2 = qvec_to_rotmat(qvec2)
    c2w = colmap_image_to_c2w(image.qvec, image.tvec, opengl=True)
    viewmat = c2w_to_viewmat(c2w)
    K = intrinsic_matrix(camera.fx, camera.fy, camera.cx, camera.cy)
    fov_x = focal_to_fov(camera.fx, camera.width)
    assert_true(rot.shape == (3, 3), "rotation matrix shape is wrong")
    assert_close(rot @ rot.T, np.eye(3), atol=1e-6, message="rotation matrix should be orthonormal")
    assert_close(abs(float(np.dot(image.qvec / np.linalg.norm(image.qvec), qvec2))), 1.0, atol=1e-6, message="qvec round trip failed")
    assert_close(rot2, rot, atol=1e-6, message="rotmat_to_qvec output must reconstruct the original rotation")
    assert_true(c2w.shape == (4, 4) and viewmat.shape == (4, 4), "pose matrix shape is wrong")
    assert_close(c2w @ viewmat, np.eye(4), atol=1e-6, message="viewmat should invert c2w")
    assert_close(viewmat[:3, :3], np.diag([1.0, -1.0, -1.0]) @ rot, atol=1e-6, message="OpenGL c2w conversion rotation does not match COLMAP convention")
    assert_close(K, np.array([[camera.fx, 0, camera.cx], [0, camera.fy, camera.cy], [0, 0, 1]], dtype=np.float64), message="K mismatch")
    assert_close(fov_to_focal(fov_x, camera.width), camera.fx, message="focal/FOV round trip failed")
    assert_true(estimate_scene_extent(COLMAP_SCENE.c2w_matrices) > 0, "scene extent must be positive")
    assert_true(average_pose_center(COLMAP_SCENE.c2w_matrices).shape == (3,), "average center shape is wrong")


def test_ply_io():
    from modules.gaussian_model import GaussianModel
    from utils.ply_io import gaussians_to_ply_dict, ply_dict_to_gaussians, read_ply, write_ply

    model = GaussianModel.from_point_cloud(TEST_XYZ[:16], TEST_RGB[:16], device="cpu")
    payload = gaussians_to_ply_dict(
        model.means.detach().numpy(),
        model.log_scales.detach().numpy(),
        model.quats.detach().numpy(),
        model.logit_opacities.detach().numpy(),
        model.features_dc.detach().numpy(),
        model.features_rest.detach().numpy(),
    )
    path = Path(tempfile.gettempdir()) / "sjtu_curasao_gaussian_test.ply"
    write_ply(str(path), payload)
    restored = ply_dict_to_gaussians(read_ply(str(path)))
    original = (
        model.means.detach().numpy(),
        model.log_scales.detach().numpy(),
        model.quats.detach().numpy(),
        model.logit_opacities.detach().numpy(),
        model.features_dc.detach().numpy(),
        model.features_rest.detach().numpy(),
    )
    assert_true([x.shape for x in restored] == [x.shape for x in original], "PLY restored shapes differ")
    for index, (restored_array, original_array) in enumerate(zip(restored, original)):
        assert_close(restored_array, original_array, atol=1e-6, message=f"PLY array {index} changed")
    return str(path)


run_test("utils syntax", test_syntax_utils)
run_test("utils.colmap_reader", test_colmap_reader)
run_test("utils.dataset_loaders", test_dataset_loaders)
run_test("utils.image_utils", test_image_utils)
run_test("utils.camera_utils", test_camera_utils)
run_test("utils.ply_io", test_ply_io)

PASS utils syntax
PASS utils.colmap_reader: 21 images, 25837 points
PASS utils.dataset_loaders: colmap=1776x1182, llff=1776x1182
PASS utils.image_utils: (1182, 1776, 3)
PASS utils.camera_utils
PASS utils.ply_io: /tmp/sjtu_curasao_gaussian_test.ply


## modules/ tests

In [29]:
def test_syntax_modules():
    for path in sorted(Path("modules").glob("*.py")):
        compile(path.read_text(), str(path), "exec")
    import modules
    import utils
    assert_true(hasattr(modules, "GaussianModel") and hasattr(modules, "GaussianRenderer"), "modules __init__ must export core assembly classes")
    assert_true(hasattr(modules, "build_3dgs_optimizer") and hasattr(modules, "photometric_loss"), "modules __init__ must export optimizer/loss helpers")
    assert_true(hasattr(utils, "load_colmap_dataset") and hasattr(utils, "load_llff_dataset"), "utils __init__ must export dataset loaders")
    from modules.densification import DensificationConfig, DensificationStats
    from modules.gaussian_model import GaussianModel
    from modules.optim import OptimConfig
    assert_true(DensificationConfig().interval == 100 and DensificationStats().total == 0, "densification dataclass defaults changed")
    assert_true(OptimConfig().position_lr > 0 and OptimConfig().eps > 0, "optimizer config defaults must be positive")
    try:
        GaussianModel(sh_degree=-1)
    except ValueError:
        pass
    else:
        raise AssertionError("GaussianModel must reject negative sh_degree")


def build_cpu_model(count=64):
    from modules.gaussian_model import GaussianModel

    return GaussianModel.from_point_cloud(TEST_XYZ[:count], TEST_RGB[:count], device="cpu")


def initialize_adam_state(model, optimizer):
    optimizer.zero_grad(set_to_none=True)
    loss = sum(param.sum() * 0.0 for param in model.parameter_map().values())
    loss.backward()
    optimizer.step()


def assert_optimizer_param_lengths(model, optimizer):
    lengths = [group["params"][0].shape[0] for group in optimizer.param_groups]
    assert_true(all(length == model.num_gaussians for length in lengths), f"optimizer param lengths mismatch: {lengths}")
    for group in optimizer.param_groups:
        param = group["params"][0]
        state = optimizer.state.get(param, {})
        for key, value in state.items():
            if torch.is_tensor(value) and value.ndim > 0:
                assert_true(value.shape[0] == model.num_gaussians, f"optimizer state {group.get('name')}:{key} length mismatch")


def test_spherical_harmonics():
    from modules.spherical_harmonics import num_sh_bases, rgb_to_sh, sh_to_rgb

    rgb = torch.tensor([[0.2, 0.4, 0.6], [1.0, 0.0, 0.5]], dtype=torch.float32)
    assert_true(num_sh_bases(0) == 1 and num_sh_bases(3) == 16, "SH basis count is wrong")
    assert_true(torch.allclose(sh_to_rgb(rgb_to_sh(rgb)), rgb, atol=1e-6), "RGB/SH round trip failed")


def test_gaussian_model():
    from modules.spherical_harmonics import sh_to_rgb
    from modules.spherical_harmonics import num_sh_bases

    model = build_cpu_model(32)
    original_means = model.means.detach().clone()
    original_log_scales = model.log_scales.detach().clone()
    original_quats = model.quats.detach().clone()
    original_opacities = model.logit_opacities.detach().clone()
    original_features_dc = model.features_dc.detach().clone()
    original_features_rest = model.features_rest.detach().clone()
    assert_true(model.num_gaussians == 32, "initial Gaussian count is wrong")
    assert_true(model.means.shape == (32, 3), "means shape is wrong")
    assert_true(model.log_scales.shape == (32, 3), "log_scales shape is wrong")
    assert_true(model.quats.shape == (32, 4), "quats shape is wrong")
    assert_true(model.colors.shape == (32, num_sh_bases(model.sh_degree), 3), "colors shape is wrong")
    assert_true(torch.all(model.scales > 0), "scales must be positive")
    assert_true(torch.allclose(model.normalized_quats.norm(dim=-1), torch.ones(32), atol=1e-6), "quats must normalize")
    assert_true(torch.all((model.opacities > 0) & (model.opacities < 1)), "opacities must be in (0, 1)")
    expected_rgb = torch.as_tensor(TEST_RGB[:32], dtype=torch.float32) / 255.0
    restored_rgb = sh_to_rgb(model.features_dc[:, 0, :].detach())
    assert_true(torch.allclose(restored_rgb, expected_rgb, atol=1e-6), "features_dc must encode input RGB exactly")
    assert_true(torch.allclose(model.opacities, torch.full_like(model.opacities, 0.1), atol=1e-6), "initial opacity must match default 0.1")
    tensors = model.activated_tensors()
    assert_true(tensors.means.shape == model.means.shape and tensors.colors.shape == model.colors.shape, "activated tensor shape mismatch")
    assert_true(set(model.parameter_map()) == {"means", "log_scales", "quats", "logit_opacities", "features_dc", "features_rest"}, "parameter_map keys mismatch")
    replacement = {name: param.detach().clone() for name, param in model.parameter_map().items()}
    replacement["means"] = replacement["means"] + 1.0
    model.replace_tensors(replacement)
    assert_true(torch.allclose(model.means, original_means + 1.0), "replace_tensors must replace means exactly")
    append_payload = {name: param.detach()[:2].clone() for name, param in model.parameter_map().items()}
    appended = model.append_tensors(append_payload)
    assert_true(appended == 2 and model.num_gaussians == 34, "append_tensors failed")
    model.replace_tensors({
        "means": original_means,
        "log_scales": original_log_scales,
        "quats": original_quats,
        "logit_opacities": original_opacities,
        "features_dc": original_features_dc,
        "features_rest": original_features_rest,
    })
    means2d = torch.zeros(model.num_gaussians, 2, requires_grad=True)
    means2d.sum().backward()
    visibility = torch.zeros(model.num_gaussians, dtype=torch.bool)
    visibility[:4] = True
    model.accumulate_gradient_stats(means2d, visibility=visibility, use_absgrad=False)
    assert_true(torch.all(model.gradient_count[:4] == 1) and torch.all(model.gradient_count[4:] == 0), "accumulate_gradient_stats visibility handling failed")
    model.clear_gradient_stats()
    assert_true(torch.all(model.gradient_accum == 0) and torch.all(model.gradient_count == 0), "clear_gradient_stats failed")
    packed_means2d = torch.zeros(3, 2, requires_grad=True)
    packed_means2d.sum().backward()
    packed_indices = torch.tensor([2, 5, 5])
    model.accumulate_gradient_stats(packed_means2d, use_absgrad=False, indices=packed_indices)
    assert_true(model.gradient_count[2].item() == 1 and model.gradient_count[5].item() == 2, "packed gaussian_ids gradient accumulation failed")
    model.clear_gradient_stats()

    clone_mask = torch.zeros(model.num_gaussians, dtype=torch.bool)
    clone_mask[:3] = True
    cloned = model.clone(clone_mask)
    assert_true(cloned == 3 and model.num_gaussians == 35, "clone failed")
    assert_true(torch.allclose(model.means[-3:], original_means[:3]), "clone means must copy selected parents")
    assert_true(torch.allclose(model.log_scales[-3:], original_log_scales[:3]), "clone scales must copy selected parents")
    assert_true(torch.allclose(model.quats[-3:], original_quats[:3]), "clone quats must copy selected parents")
    assert_true(torch.allclose(model.logit_opacities[-3:], original_opacities[:3]), "clone opacities must copy selected parents")
    assert_true(torch.allclose(model.features_dc[-3:], original_features_dc[:3]), "clone features_dc must copy selected parents")
    assert_true(torch.allclose(model.features_rest[-3:], original_features_rest[:3]), "clone features_rest must copy selected parents")

    split_mask = torch.zeros(model.num_gaussians, dtype=torch.bool)
    split_mask[:2] = True
    before_split_count = model.num_gaussians
    added, keep = model.split(split_mask, num_splits=2)
    assert_true(added == 4 and keep.shape == (before_split_count + added,) and model.num_gaussians == before_split_count - 2 + added, "split failed")
    assert_true(torch.all(model.log_scales[-4:] <= original_log_scales[:2].max() + 1e-6), "split children scales should be shrunk from parents")

    remove_mask = torch.zeros(model.num_gaussians, dtype=torch.bool)
    remove_mask[:5] = True
    keep = model.prune(remove_mask)
    assert_true(keep.sum().item() == 32 and model.num_gaussians == 32, "prune failed")
    model.reset_opacities(0.02)
    assert_true(torch.allclose(model.opacities, torch.full_like(model.opacities, 0.02), atol=1e-6), "reset_opacities failed")
    return f"{model.num_gaussians} gaussians"


def test_camera_module():
    from modules.camera import Camera
    from utils.image_utils import load_image

    camera = Camera.from_scene_data(COLMAP_SCENE, 0, device="cpu", load_image=True)
    assert_true(camera.width == COLMAP_SCENE.width and camera.height == COLMAP_SCENE.height, "camera resolution mismatch")
    assert_true(camera.image is not None and camera.image.shape == (3, camera.height, camera.width), "camera image shape is wrong")
    assert_true(camera.K.shape == (3, 3) and camera.viewmat.shape == (4, 4), "camera matrix shape is wrong")
    assert_true(camera.camera_center.shape == (3,), "camera center shape is wrong")
    expected_K = torch.tensor([[COLMAP_SCENE.fx[0], 0.0, COLMAP_SCENE.cx[0]], [0.0, COLMAP_SCENE.fy[0], COLMAP_SCENE.cy[0]], [0.0, 0.0, 1.0]], dtype=torch.float32)
    expected_image = torch.as_tensor(load_image(COLMAP_SCENE.image_paths[0], as_float=True), dtype=torch.float32).permute(2, 0, 1).contiguous()
    assert_true(torch.allclose(camera.K, expected_K, atol=1e-6), "Camera.K values must match SceneData intrinsics")
    assert_true(torch.allclose(camera.c2w, torch.as_tensor(COLMAP_SCENE.c2w_matrices[0], dtype=torch.float32), atol=1e-6), "Camera c2w must match SceneData pose")
    assert_true(torch.allclose(camera.image, expected_image, atol=0.0), "Camera image tensor must match loaded GT image")
    ident = camera.c2w @ camera.viewmat
    assert_true(torch.allclose(ident, torch.eye(4), atol=1e-5), "viewmat must invert c2w")
    camera_no_image = Camera.from_scene_data(COLMAP_SCENE, 0, device="cpu", load_image=False)
    assert_true(camera_no_image.image is None, "load_image=False should not load GT image")


def test_losses():
    from modules.losses import l1_loss, photometric_loss, ssim
    from utils.image_utils import load_image

    patch_np = load_image(str(IMAGE_PATH), as_float=True, resize=(64, 48))[:32, :32]
    patch = torch.as_tensor(patch_np, dtype=torch.float32).permute(2, 0, 1).contiguous()
    noisy = torch.clamp(patch + 0.05, 0.0, 1.0)
    assert_true(float(l1_loss(patch, patch)) == 0.0, "identical L1 should be zero")
    assert_true(float(ssim(patch, patch)) > 0.999, "identical SSIM should be near 1")
    loss_same, parts_same = photometric_loss(patch, patch)
    loss_noisy, parts_noisy = photometric_loss(noisy, patch)
    expected_l1 = torch.mean(torch.abs(noisy - patch))
    expected_dssim = (1.0 - parts_noisy["ssim"]) * 0.5
    expected_total = 0.8 * expected_l1 + 0.2 * expected_dssim
    assert_true(float(loss_same) < 1e-6, "identical photometric loss should be near zero")
    assert_true(float(loss_noisy) > float(loss_same), "noisy image loss should be larger")
    assert_true(set(parts_same) == {"l1", "ssim", "dssim", "total"}, "loss parts keys are wrong")
    assert_true(set(parts_noisy) == {"l1", "ssim", "dssim", "total"}, "loss parts keys are wrong")
    assert_true(torch.allclose(parts_noisy["l1"], expected_l1), "L1 part must equal mean absolute error")
    assert_true(torch.allclose(parts_noisy["dssim"], expected_dssim), "DSSIM part must equal (1 - SSIM) / 2")
    assert_true(torch.allclose(loss_noisy, expected_total), "photometric total must match weighted formula")


def test_optim():
    from modules.optim import OptimConfig, build_3dgs_optimizer, exponential_lr, set_group_lr

    model = build_cpu_model(16)
    config = OptimConfig(position_lr=0.01, feature_lr=0.02)
    optimizer = build_3dgs_optimizer(model, config)
    names = [group.get("name") for group in optimizer.param_groups]
    assert_true(names == ["means", "features_dc", "features_rest", "logit_opacities", "log_scales", "quats"], "optimizer group names mismatch")
    assert_true(optimizer.param_groups[0]["lr"] == 0.01, "position lr mismatch")
    assert_true(optimizer.param_groups[2]["lr"] == 0.001, "features_rest lr should be feature_lr / 20")
    set_group_lr(optimizer, "means", 0.123)
    assert_true(optimizer.param_groups[0]["lr"] == 0.123, "set_group_lr failed")
    schedule = exponential_lr(0.1, 0.01, max_steps=10, delay_steps=2, delay_mult=0.5)
    schedule_no_delay = exponential_lr(0.1, 0.01, max_steps=10)
    assert_true(schedule(-1) == 0.0, "negative step lr should be zero")
    assert_true(schedule(0) > schedule(10), "lr schedule should decay")
    assert_close(schedule_no_delay(0), 0.1, message="lr schedule step 0 must equal lr_init without delay")
    assert_close(schedule_no_delay(10), 0.01, message="lr schedule max step must equal lr_final")


def test_densification():
    from modules.densification import DensificationConfig, DensificationController
    from modules.optim import build_3dgs_optimizer
    from modules.renderer import RenderOutput

    model = build_cpu_model(16)
    optimizer = build_3dgs_optimizer(model)
    initialize_adam_state(model, optimizer)

    means2d = torch.zeros(model.num_gaussians, 2, requires_grad=True)
    means2d.sum().backward()
    output = RenderOutput(
        image=torch.empty(3, 1, 1),
        alpha=torch.empty(1, 1, 1),
        depth=None,
        radii=torch.ones(model.num_gaussians),
        means2d=means2d,
        metadata={},
    )
    controller = DensificationController(
        DensificationConfig(start_step=0, stop_step=10, interval=1, grad_threshold=0.1, scene_extent=1.0, percent_dense=999.0, min_opacity=0.0)
    )
    stats = controller.update(model, output, optimizer, step=1)
    assert_true(stats.cloned == 16 and model.num_gaussians == 32, "densification clone failed")
    assert_true(torch.allclose(model.gradient_accum, torch.zeros_like(model.gradient_accum)), "densification must clear gradient_accum after update")
    assert_true(torch.allclose(model.gradient_count, torch.zeros_like(model.gradient_count)), "densification must clear gradient_count after update")
    assert_optimizer_param_lengths(model, optimizer)

    prune_model = build_cpu_model(8)
    prune_optimizer = build_3dgs_optimizer(prune_model)
    initialize_adam_state(prune_model, prune_optimizer)
    prune_model.reset_opacities(0.001)
    prune_output = RenderOutput(
        image=torch.empty(3, 1, 1),
        alpha=torch.empty(1, 1, 1),
        depth=None,
        radii=torch.ones(prune_model.num_gaussians),
        means2d=None,
        metadata={},
    )
    prune_controller = DensificationController(
        DensificationConfig(start_step=0, stop_step=10, interval=1, grad_threshold=999.0, min_opacity=0.005)
    )
    prune_stats = prune_controller.update(prune_model, prune_output, prune_optimizer, step=1)
    assert_true(prune_stats.pruned == 8 and prune_model.num_gaussians == 0, "densification prune failed")
    assert_optimizer_param_lengths(prune_model, prune_optimizer)

    split_model = build_cpu_model(8)
    split_optimizer = build_3dgs_optimizer(split_model)
    initialize_adam_state(split_model, split_optimizer)
    with torch.no_grad():
        split_model.log_scales.fill_(0.0)
    split_means2d = torch.zeros(split_model.num_gaussians, 2, requires_grad=True)
    split_means2d.sum().backward()
    split_output = RenderOutput(
        image=torch.empty(3, 1, 1),
        alpha=torch.empty(1, 1, 1),
        depth=None,
        radii=torch.ones(split_model.num_gaussians),
        means2d=split_means2d,
        metadata={},
    )
    split_controller = DensificationController(
        DensificationConfig(start_step=0, stop_step=10, interval=1, grad_threshold=0.1, scene_extent=1.0, percent_dense=0.01, min_opacity=0.0, num_splits=2)
    )
    split_stats = split_controller.update(split_model, split_output, split_optimizer, step=1)
    assert_true(split_stats.split == 16 and split_model.num_gaussians == 16, "densification split path failed")
    assert_optimizer_param_lengths(split_model, split_optimizer)

    reset_model = build_cpu_model(4)
    reset_optimizer = build_3dgs_optimizer(reset_model)
    initialize_adam_state(reset_model, reset_optimizer)
    reset_controller = DensificationController(DensificationConfig(start_step=10, opacity_reset_interval=1, reset_opacity=0.03))
    reset_output = RenderOutput(torch.empty(3, 1, 1), torch.empty(1, 1, 1), None, None, None, {})
    reset_stats = reset_controller.update(reset_model, reset_output, reset_optimizer, step=1)
    assert_true(reset_stats.opacity_reset and torch.allclose(reset_model.opacities, torch.full_like(reset_model.opacities, 0.03), atol=1e-6), "opacity reset interval failed")


def test_renderer_optional():
    if os.environ.get("RUN_RENDERER_TEST") != "1":
        raise SkipTest("set RUN_RENDERER_TEST=1 to run the long CUDA/gsplat smoke test")
    if not torch.cuda.is_available():
        raise SkipTest("CUDA is not available")
    if importlib.util.find_spec("gsplat") is None:
        raise SkipTest("gsplat is not installed")

    from modules.camera import Camera
    from modules.gaussian_model import GaussianModel
    from modules.renderer import GaussianRenderer

    device = torch.device("cuda")
    model = GaussianModel.from_point_cloud(TEST_XYZ[:64], TEST_RGB[:64], device=device)
    scale = 32
    camera = Camera(
        width=max(COLMAP_SCENE.width // scale, 1),
        height=max(COLMAP_SCENE.height // scale, 1),
        fx=float(COLMAP_SCENE.fx[0] / scale),
        fy=float(COLMAP_SCENE.fy[0] / scale),
        cx=float(COLMAP_SCENE.cx[0] / scale),
        cy=float(COLMAP_SCENE.cy[0] / scale),
        c2w=torch.as_tensor(COLMAP_SCENE.c2w_matrices[0], dtype=torch.float32, device=device),
        near=float(COLMAP_SCENE.near),
        far=float(COLMAP_SCENE.far),
    )
    white_renderer = GaussianRenderer(background=(1.0, 1.0, 1.0))
    black_renderer = GaussianRenderer(background=(0.0, 0.0, 0.0))
    output = white_renderer.render(model, camera)
    black_output = black_renderer.render(model, camera)
    depth_output = black_renderer.render(model, camera, render_mode="RGB+D")
    assert_true(output.image.shape == (3, camera.height, camera.width), "rendered image shape is wrong")
    assert_true(output.alpha.shape == (1, camera.height, camera.width), "rendered alpha shape is wrong")
    assert_true(depth_output.depth is not None and depth_output.depth.shape == (1, camera.height, camera.width), "RGB+D render must return depth channel")
    assert_true(torch.isfinite(output.image).all(), "rendered image contains non-finite values")
    assert_true(torch.isfinite(output.alpha).all(), "rendered alpha contains non-finite values")
    assert_true(torch.all((output.alpha >= 0.0) & (output.alpha <= 1.0)), "alpha must be in [0, 1]")
    expected_background_delta = (1.0 - output.alpha).expand_as(output.image)
    assert_true(torch.allclose(output.alpha, black_output.alpha, atol=1e-5), "background color must not change alpha")
    assert_true(torch.allclose(output.image - black_output.image, expected_background_delta, atol=1e-4), "white-vs-black render delta must equal 1 - alpha")
    return f"{camera.width}x{camera.height}"


run_test("modules syntax", test_syntax_modules)
run_test("modules.spherical_harmonics", test_spherical_harmonics)
run_test("modules.gaussian_model", test_gaussian_model)
run_test("modules.camera", test_camera_module)
run_test("modules.losses", test_losses)
run_test("modules.optim", test_optim)
run_test("modules.densification", test_densification)
run_test("modules.renderer optional", test_renderer_optional)

PASS modules syntax
PASS modules.spherical_harmonics
PASS modules.gaussian_model: 32 gaussians
PASS modules.camera
PASS modules.losses
PASS modules.optim
PASS modules.densification
SKIP modules.renderer optional: set RUN_RENDERER_TEST=1 to run the long CUDA/gsplat smoke test


## Summary

In [30]:
from collections import Counter
from IPython.display import Markdown, display

counts = Counter(status for _, status, _ in RESULTS)
print("\nSummary:", dict(counts))
print("-" * 88)
for name, status, detail in RESULTS:
    short = detail.splitlines()[0] if detail else ""
    print(f"{status:4} | {name:32} | {short}")

failures = [(name, detail) for name, status, detail in RESULTS if status == "FAIL"]
if failures:
    print("\nFailures:")
    for name, detail in failures:
        print(f"\n{name}\n{detail}")
    raise AssertionError(f"{len(failures)} test(s) failed")

success_message = "✅ 全部必测项通过 / All required tests passed"
print(f"\n{success_message}. Optional skips are listed above.")
display(Markdown("""
<div style="border: 2px solid #16a34a; background: #dcfce7; color: #14532d; padding: 14px 16px; border-radius: 8px; font-size: 20px; font-weight: 700;">
✅ 全部必测项通过 / All required tests passed
</div>
"""))


Summary: {'PASS': 13, 'SKIP': 1}
----------------------------------------------------------------------------------------
PASS | utils syntax                     | 
PASS | utils.colmap_reader              | 21 images, 25837 points
PASS | utils.dataset_loaders            | colmap=1776x1182, llff=1776x1182
PASS | utils.image_utils                | (1182, 1776, 3)
PASS | utils.camera_utils               | 
PASS | utils.ply_io                     | /tmp/sjtu_curasao_gaussian_test.ply
PASS | modules syntax                   | 
PASS | modules.spherical_harmonics      | 
PASS | modules.gaussian_model           | 32 gaussians
PASS | modules.camera                   | 
PASS | modules.losses                   | 
PASS | modules.optim                    | 
PASS | modules.densification            | 
SKIP | modules.renderer optional        | set RUN_RENDERER_TEST=1 to run the long CUDA/gsplat smoke test

✅ 全部必测项通过 / All required tests passed. Optional skips are listed above.



<div style="border: 2px solid #16a34a; background: #dcfce7; color: #14532d; padding: 14px 16px; border-radius: 8px; font-size: 20px; font-weight: 700;">
✅ 全部必测项通过 / All required tests passed
</div>
